# TP — Séance 1 : Fondamentaux des LLM
### Master IDD · IA générative & LLM — IAE Grenoble

Ce notebook accompagne la Séance 1 du cours. Il contient les **trois TP** de la séance :

1. **TP1 — Comparaison de plusieurs modèles** (~30 min)
2. **TP2 — Analyse des hallucinations** (~30 min)
3. **TP3 — Prompts & appel API** (~30 min)

**Outils utilisés (100% gratuits) :** Google Colab, [Hugging Face Hub](https://huggingface.co/) (Inference API), `pandas`.

> 💡 Ouvrez ce notebook dans **Google Colab** (`Fichier > Ouvrir dans Colab` si vous êtes sur GitHub, ou glissez-déposez le fichier sur [colab.research.google.com](https://colab.research.google.com)).

## 0. Préparation

### 0.1 Créer un compte et un token Hugging Face (gratuit)

1. Créez un compte sur [huggingface.co](https://huggingface.co/join) (gratuit).
2. Allez dans **Settings → Access Tokens** et créez un token de type **Read**.
3. Collez-le ci-dessous quand on vous le demandera (il ne sera pas affiché en clair).

Nous utiliserons des **modèles ouverts** (Llama, Mistral, Qwen…) via l'API d'inférence Hugging Face pour la partie exécutable.
Pour les modèles propriétaires (**GPT, Claude, Gemini**), qui nécessitent des clés payantes, vous les testerez manuellement via leurs interfaces web gratuites (voir consignes TP1) et consignerez vos observations dans les tableaux prévus.

In [1]:
!pip install -q huggingface_hub pandas

In [2]:
import os
import time
import pandas as pd
from dotenv import load_dotenv
from google.colab import userdata
from huggingface_hub import InferenceClient

In [ ]:
load_dotenv(".env")

True

In [3]:
# Collez votre token Hugging Face (Settings > Access Tokens) — il reste masqué
# HF_TOKEN = os.environ["HF_TOKEN"]
HF_TOKEN = userdata.get('HF_TOKEN')

client = InferenceClient(token=HF_TOKEN)

# Modèles ouverts utilisés pour le TP (vous pouvez en substituer un si l'un est indisponible)
OPEN_MODELS = {
    "Llama 3.1 8B Instruct": "meta-llama/Llama-3.1-8B-Instruct",
    "Mistral 7B Instruct": "mistralai/Mistral-7B-Instruct-v0.3",
    "Qwen 2.5 7B Instruct": "Qwen/Qwen2.5-7B-Instruct",
}


def ask_model(prompt, model_id, max_tokens=300):
    """Interroge un modèle ouvert via l'API d'inférence Hugging Face.
    Renvoie (réponse, temps_de_réponse_en_secondes) ou un message d'erreur explicite."""
    start = time.time()
    try:
        response = client.chat_completion(
            messages=[{"role": "user", "content": prompt}],
            model=model_id,
            max_tokens=max_tokens,
        )
        elapsed = round(time.time() - start, 2)
        return response.choices[0].message.content, elapsed
    except Exception as e:
        elapsed = round(time.time() - start, 2)
        return f"[Erreur / modèle indisponible : {e}]", elapsed


print("Configuration prête. Modèles disponibles :", list(OPEN_MODELS.keys()))

Configuration prête. Modèles disponibles : ['Llama 3.1 8B Instruct', 'Mistral 7B Instruct', 'Qwen 2.5 7B Instruct']


---
## TP1 — Comparaison de plusieurs modèles (~30 min)

**Objectif :** sur un même jeu de prompts, interroger plusieurs modèles et comparer les réponses obtenues (exactitude, clarté, créativité, rapidité).

### Consignes
- Testez chaque prompt sur au moins 3 modèles.
- Notez les différences de style, de longueur, de précision.
- Relevez les temps de réponse.

In [4]:
# 1.1 — Choisissez 3 prompts variés : factuel, créatif, raisonnement
prompts = {
    "Factuel": "Quelle est la capitale de l'Australie et depuis quand le est-elle ?",
    "Créatif": "Écris un slogan pour une application mobile qui aide les étudiants à réviser leurs examens.",
    "Raisonnement": "Un train part de Grenoble à 8h00 à 90 km/h. Un second part de Lyon (110 km plus loin) "
    "à 8h15 à 110 km/h en direction de Grenoble. À quelle heure se croisent-ils ?",
}
for k, v in prompts.items():
    print(f"[{k}] {v}\n")

[Factuel] Quelle est la capitale de l'Australie et depuis quand le est-elle ?

[Créatif] Écris un slogan pour une application mobile qui aide les étudiants à réviser leurs examens.

[Raisonnement] Un train part de Grenoble à 8h00 à 90 km/h. Un second part de Lyon (110 km plus loin) à 8h15 à 110 km/h en direction de Grenoble. À quelle heure se croisent-ils ?



In [5]:
# 1.2 — Interroger les modèles ouverts sur chaque prompt
results = []
for prompt_type, prompt_text in prompts.items():
    for model_name, model_id in OPEN_MODELS.items():
        answer, elapsed = ask_model(prompt_text, model_id)
        results.append(
            {
                "Type de prompt": prompt_type,
                "Modèle": model_name,
                "Réponse": answer,
                "Temps (s)": elapsed,
            }
        )
        print(f"✓ {model_name} — {prompt_type} — {elapsed}s")

df_results = pd.DataFrame(results)
df_results

✓ Llama 3.1 8B Instruct — Factuel — 1.03s
✓ Mistral 7B Instruct — Factuel — 0.02s
✓ Qwen 2.5 7B Instruct — Factuel — 0.02s
✓ Llama 3.1 8B Instruct — Créatif — 5.8s
✓ Mistral 7B Instruct — Créatif — 0.02s
✓ Qwen 2.5 7B Instruct — Créatif — 0.02s
✓ Llama 3.1 8B Instruct — Raisonnement — 8.42s
✓ Mistral 7B Instruct — Raisonnement — 0.02s
✓ Qwen 2.5 7B Instruct — Raisonnement — 0.02s


,Type de prompt,Modèle,Réponse,Temps (s)
0,Factuel,Llama 3.1 8B Instruct,La capitale de l'Australie est Canberra. Elle ...,1.03
1,Factuel,Mistral 7B Instruct,[Erreur / modèle indisponible : (Request ID: R...,0.02
2,Factuel,Qwen 2.5 7B Instruct,[Erreur / modèle indisponible : (Request ID: R...,0.02
3,Créatif,Llama 3.1 8B Instruct,Voici quelques idées de slogans pour une appli...,5.80
4,Créatif,Mistral 7B Instruct,[Erreur / modèle indisponible : (Request ID: R...,0.02
5,Créatif,Qwen 2.5 7B Instruct,[Erreur / modèle indisponible : (Request ID: R...,0.02
6,Raisonnement,Llama 3.1 8B Instruct,## Étape 1 : Déterminer le temps nécessaire au...,8.42
7,Raisonnement,Mistral 7B Instruct,[Erreur / modèle indisponible : (Request ID: R...,0.02
8,Raisonnement,Qwen 2.5 7B Instruct,[Erreur / modèle indisponible : (Request ID: R...,0.02


In [7]:
# 1.3 — Affichage détaillé d'une réponse (pratique pour la comparer en entier)
# Changez les valeurs ci-dessous pour explorer une réponse précise
type_a_afficher = "Raisonnement"
modele_a_afficher = "Llama 3.1 8B Instruct"

ligne = df_results[
    (df_results["Type de prompt"] == type_a_afficher)
    & (df_results["Modèle"] == modele_a_afficher)
]
print(
    ligne["Réponse"].values[0]
    if len(ligne)
    else "Combinaison introuvable — vérifiez les noms ci-dessus."
)

## Étape 1 : Déterminer le temps nécessaire au premier train pour parcourir la distance entre Lyon et Grenoble.
Le premier train parcourt 110 km à 90 km/h. Pour trouver le temps nécessaire, nous utilisons la formule temps = distance / vitesse. Ainsi, le temps nécessaire au premier train est de 110 km / 90 km/h = 1,22 heures.

## Étape 2 : Déterminer le temps nécessaire au deuxième train pour parcourir la distance entre Lyon et Grenoble.
Le deuxième train parcourt la même distance (110 km) mais à 110 km/h. Pour trouver le temps nécessaire, nous utilisons à nouveau la formule temps = distance / vitesse. Ainsi, le temps nécessaire au deuxième train est de 110 km / 110 km/h = 1 heure.

## Étape 3 : Calculer le temps pendant lequel les deux trains se déplacent simultanément.
Le premier train part à 8h00 et le deuxième train part à 8h15, donc les deux trains se déplacent simultanément pendant 15 minutes.

## Étape 4 : Calculer le temps nécessaire au premier train pour parcourir la distance j

### 1.4 — Modèles propriétaires (GPT, Claude, Gemini)

Ces modèles ne sont pas gratuits via API. Testez-les **manuellement**, avec les mêmes 3 prompts, via leurs interfaces gratuites :

| Modèle | Interface gratuite |
|---|---|
| GPT | [chatgpt.com](https://chatgpt.com) |
| Claude | [claude.ai](https://claude.ai) |
| Gemini | [gemini.google.com](https://gemini.google.com) |

Consignez vos observations dans le tableau ci-dessous (exécutez la cellule puis éditez les valeurs).

In [ ]:
# 1.5 — Grille de comparaison à compléter manuellement (modèles propriétaires)
grille_manuelle = pd.DataFrame(
    {
        "Critère": ["Exactitude", "Clarté", "Créativité", "Rapidité perçue"],
        "GPT": ["", "", "", ""],
        "Claude": ["", "", "", ""],
        "Gemini": ["", "", "", ""],
    }
)
# Exemple pour éditer une cellule : grille_manuelle.loc[0, "GPT"] = "5/5 - réponse correcte et sourcée"
grille_manuelle

### 💬 Questions de synthèse TP1
- Quel modèle a été le plus rapide ? Le plus précis sur le prompt de raisonnement ?
- Avez-vous observé de grandes différences de style entre modèles ouverts et propriétaires ?
- Sur quel(s) critère(s) choisiriez-vous un modèle plutôt qu'un autre pour un usage professionnel ?

---
## TP2 — Analyse des hallucinations (~30 min)

**Objectif :** apprendre à **provoquer**, **repérer** et **contrer** une hallucination — une réponse formulée avec assurance mais factuellement fausse ou inventée.

In [ ]:
# 2.1 — Prompts conçus pour provoquer une hallucination
prompts_pieges = [
    "Cite l'article exact et le numéro de page où Victor Hugo mentionne l'intelligence artificielle dans Les Misérables.",
    "Qui a remporté la Coupe du Monde de football en 2030 ?",
    "Donne-moi le titre et la date de publication du rapport officiel de l'ONU sur la population de la Lune.",
]

modele_test = "Mistral 7B Instruct"  # vous pouvez changer de modèle

reponses_hallucinations = []
for p in prompts_pieges:
    answer, elapsed = ask_model(p, OPEN_MODELS[modele_test])
    reponses_hallucinations.append(
        {"Prompt piège": p, "Modèle": modele_test, "Réponse": answer}
    )
    print(f"— {p}\n→ {answer}\n")

df_hallucinations = pd.DataFrame(reponses_hallucinations)
df_hallucinations

### 2.2 — Repérer

Pour chaque réponse obtenue ci-dessus :
- Vérifiez chaque affirmation factuelle (recherche web, source fiable).
- Cherchez les signaux de « fausse confiance » (détails inventés très précis, absence de nuance).
- Comparez avec la réponse d'un second modèle sur le même prompt.

In [ ]:
# 2.3 — Journal d'analyse à compléter
journal_hallucinations = pd.DataFrame(
    {
        "Prompt piège": [p[:40] + "…" for p in prompts_pieges],
        "Hallucination détectée ? (Oui/Non)": ["", "", ""],
        "Élément(s) inventé(s)": ["", "", ""],
        "Comment la contrer (reformulation, RAG, contrainte...)": ["", "", ""],
    }
)
journal_hallucinations

### 2.4 — Contrer

Reformulez l'un des prompts pièges pour réduire le risque d'hallucination (ex. demander explicitement une source, autoriser le modèle à dire « je ne sais pas », limiter la portée de la question).

In [ ]:
prompt_original = prompts_pieges[0]
prompt_reformule = (
    "Si tu ne connais pas la réponse avec certitude, réponds explicitement "
    "\"Je ne dispose pas d'information fiable à ce sujet\" plutôt que d'inventer une source. "
    + prompt_original
)

reponse_originale, _ = ask_model(prompt_original, OPEN_MODELS[modele_test])
reponse_reformulee, _ = ask_model(prompt_reformule, OPEN_MODELS[modele_test])

print("AVANT :\n", reponse_originale)
print("\nAPRÈS :\n", reponse_reformulee)

### 💬 Questions de synthèse TP2
- La reformulation a-t-elle réduit l'hallucination ? Pourquoi (ou pourquoi pas) selon vous ?
- Quelle autre stratégie du cours (RAG, comparaison multi-modèles…) pourrait aider ici ?

---
## TP3 — Prompts & appel API (~30 min)

**Objectif :** comprendre comment un modèle est appelé par API, puis réécrire un prompt faible et mesurer l'amélioration obtenue, à l'aide d'une grille d'évaluation.

**Grille d'évaluation (1 = absent, 5 = excellent) :** Clarté · Contexte fourni · Contraintes de format · Exemples · Rôle assigné

### 3.0 — Sous le capot : anatomie d'un appel API

Jusqu'ici, `ask_model()` a caché l'appel. Voici ce qu'il fait réellement : on envoie une liste de **messages** (`system` = consignes générales, `user` = la demande) avec quelques **paramètres**, et on récupère la réponse ainsi que le **nombre de tokens** consommés (base du coût et de la latence).

In [ ]:
# 3.0a — Un appel API complet : messages system + user, paramètres, usage en tokens
messages = [
    {"role": "system", "content": "Tu es un assistant pédagogique. Réponds en français, en 3 phrases maximum."},
    {"role": "user", "content": "Explique ce qu'est un token à un étudiant de Master en gestion."},
]

try:
    response = client.chat_completion(
        messages=messages,
        model=OPEN_MODELS["Qwen 2.5 7B Instruct"],
        max_tokens=150,   # longueur maximale de la réponse (en tokens)
        temperature=0.7,  # degré de variété de la réponse
    )
    print(response.choices[0].message.content)
    u = response.usage
    print(f"\nTokens — prompt : {u.prompt_tokens} | réponse : {u.completion_tokens} | total : {u.total_tokens}")
except Exception as e:
    print(f"[Erreur / modèle indisponible : {e}]")

In [ ]:
# 3.0b — Effet de la température : faible = réponses stables, élevée = plus variées
question = [{"role": "user", "content": "Propose un slogan pour une application de révision d'examens."}]

for temperature in (0.1, 1.2):
    print(f"--- température = {temperature} ---")
    for essai in range(1, 3):
        try:
            r = client.chat_completion(
                messages=question,
                model=OPEN_MODELS["Qwen 2.5 7B Instruct"],
                max_tokens=40,
                temperature=temperature,
            )
            print(f"  essai {essai} :", r.choices[0].message.content.strip())
        except Exception as e:
            print(f"  [Erreur : {e}]")

**À vous :** modifiez le message `system` (ex. « réponds en une seule phrase », « adopte un ton formel ») puis relancez 3.0a. Que change-t-il ? Comment le nombre de tokens évolue-t-il avec la longueur du prompt ?

Passons maintenant à l'évaluation de prompts (3.1 à 3.4).

In [ ]:
prompt_faible = "Parle-moi de l'IA"

prompt_ameliore = (
    "Tu es formateur en école de commerce. Explique l'IA générative en 3 points "
    "à des étudiants de Master non-techniques, avec un exemple concret par point."
)

modele_eval = "Qwen 2.5 7B Instruct"  # vous pouvez changer de modèle

reponse_faible, temps_faible = ask_model(prompt_faible, OPEN_MODELS[modele_eval])
reponse_amelioree, temps_amelioree = ask_model(
    prompt_ameliore, OPEN_MODELS[modele_eval]
)

print("=== Prompt faible ===")
print(reponse_faible)
print(f"\n(temps: {temps_faible}s)\n")
print("=== Prompt amélioré ===")
print(reponse_amelioree)
print(f"\n(temps: {temps_amelioree}s)")

In [ ]:
# 3.2 — Notez chaque prompt sur la grille d'évaluation (1 à 5)
grille_evaluation = pd.DataFrame(
    {
        "Critère": [
            "Clarté",
            "Contexte fourni",
            "Contraintes de format",
            "Exemples",
            "Rôle assigné",
        ],
        "Prompt faible": [None, None, None, None, None],
        "Prompt amélioré": [None, None, None, None, None],
    }
)
# Exemple : grille_evaluation.loc[0, "Prompt faible"] = 1
grille_evaluation

In [ ]:
# 3.3 — Score total (à exécuter une fois la grille remplie ci-dessus)
if grille_evaluation[["Prompt faible", "Prompt amélioré"]].isnull().values.any():
    print(
        "⚠️  Complétez d'abord la grille ci-dessus (remplacez les None par une note de 1 à 5)."
    )
else:
    totaux = grille_evaluation[["Prompt faible", "Prompt amélioré"]].sum()
    print("Score total sur 25 :")
    print(totaux)

### 3.4 — À votre tour

Choisissez un prompt faible de votre choix (lié à un domaine qui vous intéresse : RH, marketing, finance…), améliorez-le en appliquant la grille, et comparez les deux réponses.

In [ ]:
mon_prompt_faible = ""  # à compléter
mon_prompt_ameliore = ""  # à compléter

if mon_prompt_faible and mon_prompt_ameliore:
    r1, _ = ask_model(mon_prompt_faible, OPEN_MODELS[modele_eval])
    r2, _ = ask_model(mon_prompt_ameliore, OPEN_MODELS[modele_eval])
    print("=== Votre prompt faible ===\n", r1)
    print("\n=== Votre prompt amélioré ===\n", r2)
else:
    print(
        "Complétez mon_prompt_faible et mon_prompt_ameliore ci-dessus puis relancez la cellule."
    )

### 💬 Questions de synthèse TP3
- Quel critère de la grille a le plus fait progresser la qualité de la réponse ?
- Ce résultat serait-il différent avec un autre modèle ? Pourquoi ?

---
## Synthèse de la séance

- Un LLM traite des **tokens**, pas des mots.
- Les **embeddings** encodent le sens sous forme de vecteurs.
- La **fenêtre de contexte** limite ce que le modèle peut « voir ».
- L'**attention** pondère l'importance des mots du contexte.
- Un **prompt structuré** (rôle, contexte, tâche, format) améliore nettement la réponse.
- Un modèle s'appelle par **API** (messages `system`/`user`, température, `max_tokens`) ; coût et latence se mesurent en **tokens**.
- Aucun modèle n'est infaillible : **vérifier reste indispensable**.

### Pour la Séance 2
- Créez vos comptes Google Colab et Hugging Face si ce n'est pas déjà fait.
- Relisez vos notes sur le RAG et les embeddings.
- Apportez le résultat de ce TP1 (comparaison de modèles).

**Séance 2 : RAG — construire un assistant documentaire** — Embeddings, bases vectorielles, recherche sémantique → TP : assistant documentaire (LangChain + ChromaDB + Gradio).